# 06 — Predictive Modeling Pipeline

**Objective:** Train and evaluate classification models to predict customer churn, perform hyperparameter tuning using Optuna, and persist the best performing model.

In [2]:
import sys
sys.path.insert(0, '..')

import os
import pandas as pd
import numpy as np
import joblib
from pathlib import Path
from typing import Dict, Tuple, Any, Optional

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix
)

import lightgbm as lgb
import xgboost as xgb
import optuna
from optuna.samplers import TPESampler

from src.config import *
from src.utils import logger, timer, save_model, load_model, save_csv

c:\Users\admin\OneDrive\Desktop\airline-loyality\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Data Preparation

Load engineered features and split the dataset into stratified training and testing sets.

In [3]:
def prepare_modeling_data(
    filepath: Optional[Path] = None,
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.Series, pd.Series, list]:
    if filepath is None:
        filepath = FEATURES_DIR / "customer_features.csv"

    df = pd.read_csv(filepath)
    feature_cols = [c for c in df.columns if c not in [PK, "Churn"]]

    X = df[feature_cols]
    y = df["Churn"]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=TEST_SIZE, random_state=RANDOM_SEED, stratify=y
    )

    logger.info(f"Train: {X_train.shape}, Test: {X_test.shape}")
    logger.info(f"Train churn rate: {y_train.mean():.3f}, Test churn rate: {y_test.mean():.3f}")

    return X_train, X_test, y_train, y_test, feature_cols

X_train, X_test, y_train, y_test, feature_cols = prepare_modeling_data()

2026-06-11 23:46:47 | airline_loyalty | INFO | Train: (13389, 43), Test: (3348, 43)
2026-06-11 23:46:47 | airline_loyalty | INFO | Train churn rate: 0.167, Test churn rate: 0.167


## 2. Evaluation Helper

Define a function to calculate classification performance metrics.

In [4]:
def evaluate_model(y_true, y_pred, y_prob) -> Dict[str, float]:
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_true, y_prob),
        "pr_auc": average_precision_score(y_true, y_prob),
        "confusion_matrix": confusion_matrix(y_true, y_pred),
        "y_pred": y_pred,
        "y_prob": y_prob,
    }

## 3. Model Training

Train and evaluate baseline classification algorithms: Logistic Regression, Random Forest, XGBoost, and LightGBM.

In [6]:
def train_logistic_regression(X_train, y_train, X_test, y_test) -> Dict[str, Any]:
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    model = LogisticRegression(
        class_weight="balanced",
        max_iter=1000,
        random_state=RANDOM_SEED,
        solver="lbfgs",
    )
    model.fit(X_train_scaled, y_train)

    y_pred = model.predict(X_test_scaled)
    y_prob = model.predict_proba(X_test_scaled)[:, 1]

    metrics = evaluate_model(y_test, y_pred, y_prob)
    metrics["model"] = model
    metrics["scaler"] = scaler
    metrics["name"] = "Logistic Regression"

    return metrics

def train_random_forest(X_train, y_train, X_test, y_test) -> Dict[str, Any]:
    model = RandomForestClassifier(
        n_estimators=200,
        max_depth=10,
        class_weight="balanced",
        random_state=RANDOM_SEED,
        n_jobs=-1,
    )
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    metrics = evaluate_model(y_test, y_pred, y_prob)
    metrics["model"] = model
    metrics["name"] = "Random Forest"
    metrics["feature_importance"] = dict(
        zip(X_train.columns, model.feature_importances_)
    )

    return metrics

def train_xgboost(X_train, y_train, X_test, y_test) -> Dict[str, Any]:
    scale_pos = (y_train == 0).sum() / (y_train == 1).sum()

    model = xgb.XGBClassifier(
        n_estimators=300,
        max_depth=6,
        learning_rate=0.05,
        scale_pos_weight=scale_pos,
        random_state=RANDOM_SEED,
        eval_metric="logloss",
        use_label_encoder=False,
        n_jobs=-1,
    )
    model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)

    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    metrics = evaluate_model(y_test, y_pred, y_prob)
    metrics["model"] = model
    metrics["name"] = "XGBoost"
    metrics["feature_importance"] = dict(
        zip(X_train.columns, model.feature_importances_)
    )

    return metrics

def train_lightgbm(X_train, y_train, X_test, y_test) -> Dict[str, Any]:
    model = lgb.LGBMClassifier(
        n_estimators=300,
        max_depth=6,
        learning_rate=0.05,
        is_unbalance=True,
        random_state=RANDOM_SEED,
        n_jobs=-1,
        verbose=-1,
    )
    model.fit(
        X_train, y_train,
        eval_set=[(X_test, y_test)],
        callbacks=[lgb.log_evaluation(period=0)],
    )

    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    metrics = evaluate_model(y_test, y_pred, y_prob)
    metrics["model"] = model
    metrics["name"] = "LightGBM"
    metrics["feature_importance"] = dict(
        zip(X_train.columns, model.feature_importances_)
    )

    return metrics

## 4. Hyperparameter Tuning

Perform Bayesian optimization on LightGBM using Optuna to maximize Precision-Recall AUC.

In [7]:
def tune_lightgbm(X_train, y_train, X_test, y_test, n_trials: int = 30) -> Dict[str, Any]:
    def objective(trial):
        params = {
            "n_estimators": trial.suggest_int("n_estimators", 100, 500),
            "max_depth": trial.suggest_int("max_depth", 3, 10),
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
            "num_leaves": trial.suggest_int("num_leaves", 20, 100),
            "min_child_samples": trial.suggest_int("min_child_samples", 5, 50),
            "subsample": trial.suggest_float("subsample", 0.6, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
            "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
            "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
            "is_unbalance": True,
            "random_state": RANDOM_SEED,
            "n_jobs": -1,
            "verbose": -1,
        }

        model = lgb.LGBMClassifier(**params)
        model.fit(
            X_train, y_train,
            eval_set=[(X_test, y_test)],
            callbacks=[lgb.log_evaluation(period=0), lgb.early_stopping(stopping_rounds=20, verbose=False)],
        )

        y_prob = model.predict_proba(X_test)[:, 1]
        return average_precision_score(y_test, y_prob)

    sampler = TPESampler(seed=RANDOM_SEED)
    study = optuna.create_study(direction="maximize", sampler=sampler)
    study.optimize(objective, n_trials=n_trials, show_progress_bar=False)

    logger.info(f"Best PR AUC: {study.best_value:.4f}")
    logger.info(f"Best params: {study.best_params}")

    best_params = study.best_params
    best_params["is_unbalance"] = True
    best_params["random_state"] = RANDOM_SEED
    best_params["n_jobs"] = -1
    best_params["verbose"] = -1

    final_model = lgb.LGBMClassifier(**best_params)
    final_model.fit(X_train, y_train)

    y_pred = final_model.predict(X_test)
    y_prob = final_model.predict_proba(X_test)[:, 1]

    metrics = evaluate_model(y_test, y_pred, y_prob)
    metrics["model"] = final_model
    metrics["name"] = "LightGBM (Tuned)"
    metrics["best_params"] = best_params
    metrics["feature_importance"] = dict(
        zip(X_train.columns, final_model.feature_importances_)
    )

    return metrics

## 5. Model Selection & Persistence

Compare models, choose the best one based on PR AUC, save it, and generate predictions on the full dataset.

In [8]:
def compare_models(results: list) -> pd.DataFrame:
    comparison = []
    for r in results:
        comparison.append({
            "Model": r["name"],
            "Accuracy": round(r["accuracy"], 4),
            "Precision": round(r["precision"], 4),
            "Recall": round(r["recall"], 4),
            "F1": round(r["f1"], 4),
            "ROC AUC": round(r["roc_auc"], 4),
            "PR AUC": round(r["pr_auc"], 4),
        })
    return pd.DataFrame(comparison).sort_values("PR AUC", ascending=False)

def save_best_model(best_result: Dict[str, Any], feature_cols: list):
    save_model(best_result["model"], MODELS_DIR / "churn_model.joblib")

    if "scaler" in best_result:
        save_model(best_result["scaler"], MODELS_DIR / "scaler.joblib")

    pd.Series(feature_cols).to_csv(MODELS_DIR / "feature_columns.csv", index=False)

    metadata = {
        "model_name": best_result["name"],
        "accuracy": best_result["accuracy"],
        "precision": best_result["precision"],
        "recall": best_result["recall"],
        "f1": best_result["f1"],
        "roc_auc": best_result["roc_auc"],
        "pr_auc": best_result["pr_auc"],
    }
    pd.Series(metadata).to_json(MODELS_DIR / "model_metadata.json")
    logger.info("Model artifacts saved.")

@timer
def generate_predictions(features_path: Optional[Path] = None) -> pd.DataFrame:
    if features_path is None:
        features_path = FEATURES_DIR / "customer_features.csv"
        
    df = pd.read_csv(features_path)
    model = load_model(MODELS_DIR / "churn_model.joblib")
    feature_cols = pd.read_csv(MODELS_DIR / "feature_columns.csv").iloc[:, 0].tolist()
    X = df[feature_cols]
    
    scaler_path = MODELS_DIR / "scaler.joblib"
    if scaler_path.exists():
        scaler = load_model(scaler_path)
        X_scaled = scaler.transform(X)
        probs = model.predict_proba(X_scaled)[:, 1]
    else:
        probs = model.predict_proba(X)[:, 1]
        
    predictions_df = pd.DataFrame({
        PK: df[PK],
        "Churn_Probability": probs,
        "Churn_Prediction": (probs >= 0.5).astype(int)
    })
    
    save_csv(predictions_df, FEATURES_DIR / "churn_predictions.csv")
    logger.info(f"Churn predictions saved to: {FEATURES_DIR / 'churn_predictions.csv'}")
    return predictions_df

## 6. Execution

Run the training pipeline.

In [9]:
TUNE = os.environ.get("TUNE", "True") == "True"
N_TRIALS = int(os.environ.get("N_TRIALS", "30"))

results = []
results.append(train_logistic_regression(X_train, y_train, X_test, y_test))
results.append(train_random_forest(X_train, y_train, X_test, y_test))
results.append(train_xgboost(X_train, y_train, X_test, y_test))
results.append(train_lightgbm(X_train, y_train, X_test, y_test))

if TUNE:
    results.append(tune_lightgbm(X_train, y_train, X_test, y_test, n_trials=N_TRIALS))

comparison = compare_models(results)
print(comparison)

best_result = max(results, key=lambda x: x["pr_auc"])
save_best_model(best_result, feature_cols)
generate_predictions()

c:\Users\admin\OneDrive\Desktop\airline-loyality\.venv\Lib\site-packages\xgboost\training.py:200: UserWarning: [23:48:08] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
[I 2026-06-11 23:48:09,194] A new study created in memory with name: no-name-d850899b-c6b5-46af-aff6-0be0b2dcb5bd
[I 2026-06-11 23:48:09,538] Trial 0 finished with value: 0.9362368519335668 and parameters: {'n_estimators': 250, 'max_depth': 10, 'learning_rate': 0.08960785365368121, 'num_leaves': 68, 'min_child_samples': 12, 'subsample': 0.662397808134481, 'colsample_bytree': 0.6232334448672797, 'reg_alpha': 0.6245760287469893, 'reg_lambda': 0.002570603566117598}. Best is trial 0 with value: 0.9362368519335668.
[I 2026-06-11 23:48:09,713] Trial 1 finished with value: 0.9384663775344672 and parameters: {'n_estimators': 383, 'max_depth': 3, 'learning_rate': 0.18276027831785724, 'num_leaves': 87, 'min_child_

                 Model  Accuracy  Precision  Recall      F1  ROC AUC  PR AUC
2              XGBoost    0.9591     0.8907  0.8605  0.8753   0.9653  0.9392
4     LightGBM (Tuned)    0.9585     0.8889  0.8587  0.8735   0.9648  0.9381
3             LightGBM    0.9591     0.8907  0.8605  0.8753   0.9612  0.9356
0  Logistic Regression    0.9086     0.6641  0.9159  0.7699   0.9600  0.9068
1        Random Forest    0.9456     0.8434  0.8283  0.8357   0.9490  0.9033


,Loyalty Number,Churn_Probability,Churn_Prediction
0,480934,0.030909,0
1,549612,0.155441,0
2,429460,0.982452,1
3,608370,0.009777,0
4,530508,0.023283,0
...,...,...,...
16732,823768,0.046509,0
16733,680886,0.016700,0
16734,776187,0.010634,0
16735,906428,0.070120,0
